# Analysis of Ship Noise and Ambient Sound

Erin Mee

3/7/2026

## Imports

In [2]:
from orcasound_noise.analysis.partitioned_accessor import PartitionedAccessor
from orcasound_noise.utils import Hydrophone

In [3]:
import datetime as dt
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

## Get Ship and Sound Data

In [4]:
shipdata_path = "s3://acoustic-sandbox/ambient-sound-analysis/temp_ship_metrics/year=2026/month=02/"
ship_lf = pl.scan_parquet(shipdata_path, storage_options={'aws_region': 'us-west-2'})

In [5]:
ship_df = ship_lf.collect()

In [5]:
ship_df.head()

id_track,s_timestamp,l_timestamp,duration,avg_speed,max_speed,min_speed,distance,curviness,confidence,mmsi,name,draft,type_id,type,is_isolated,comm_bb_avg,comm_bb_q05,comm_bb_q25,comm_bb_q50,comm_bb_q75,comm_bb_q95,bb_avg,bb_q05,bb_q25,bb_q50,bb_q75,bb_q95,ship_bb_avg,ship_bb_q05,ship_bb_q25,ship_bb_q50,ship_bb_q75,ship_bb_q95,bb_lsr_q50,bb_lsr_q95,comm_bb_lsr_q50,comm_bb_lsr_q95,min_dist,day
str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,str,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
"""48972891""",2026-02-01 23:12:24,2026-02-01 23:13:52,88.0,13.814615,17.7,7.6,0.773894,1.020735,0.504613,null,null,null,null,null,false,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,4539.239788,1
"""48972892""",2026-02-01 23:12:31,2026-02-01 23:39:36,1625.0,11.499823,12.5,7.9,9.875221,1.021363,0.998158,"""352001532""","""RICHWAY TRADER""",14.2,70.0,"""cargo""",false,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1630.26132,1
"""48961983""",2026-02-01 12:30:13,2026-02-01 13:00:22,1809.0,11.852797,12.9,6.8,11.261419,1.014022,0.99837,null,null,null,null,null,false,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1641.059133,1
"""48964002""",2026-02-01 13:53:23,2026-02-01 14:17:23,1440.0,17.613816,19.3,15.8,13.202136,1.021374,0.943406,null,null,null,null,null,false,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2202.067406,1
"""48952300""",2026-02-01 03:00:16,2026-02-01 03:20:07,1191.0,0.0625,1.4,0.0,0.697199,4.369674,0.991064,null,null,null,null,null,true,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,163.406462,1


In [6]:
ship_df.describe()

statistic,id_track,s_timestamp,l_timestamp,duration,avg_speed,max_speed,min_speed,distance,curviness,confidence,mmsi,name,draft,type_id,type,is_isolated,comm_bb_avg,comm_bb_q05,comm_bb_q25,comm_bb_q50,comm_bb_q75,comm_bb_q95,bb_avg,bb_q05,bb_q25,bb_q50,bb_q75,bb_q95,ship_bb_avg,ship_bb_q05,ship_bb_q25,ship_bb_q50,ship_bb_q75,ship_bb_q95,bb_lsr_q50,bb_lsr_q95,comm_bb_lsr_q50,comm_bb_lsr_q95,min_dist,day
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""1162""","""1162""","""1162""",1162.0,1162.0,1162.0,1162.0,1162.0,1162.0,1162.0,"""151""","""151""",151.0,151.0,"""151""",1162.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,544.0,1162.0,1162.0
"""null_count""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""1011""","""1011""",1011.0,1011.0,"""1011""",0.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,618.0,0.0,0.0
"""mean""",null,"""2026-02-12 23:09:16.559380""","""2026-02-12 23:34:28.014630""",1511.45525,11.785667,13.832806,8.60296,9.478036,1.234981,0.942244,null,null,10.961589,67.258278,null,0.142857,-2.955924,-7.788654,-5.401714,-2.754229,-0.564454,1.670268,-2.955924,-7.788654,-5.401714,-2.754229,-0.564454,1.670268,-2.955924,-7.788654,-5.401714,-2.754229,-0.564454,1.670268,-6.7774e23,-6.6362e23,-6.7774e23,-6.6362e23,3871.095931,12.475043
"""std""",null,null,null,2573.080731,5.229783,6.035886,5.238645,10.185012,1.606176,0.107927,null,null,3.502815,14.856408,null,null,52.818967,52.620637,52.744969,52.953783,53.636082,53.722608,52.818967,52.620637,52.744969,52.953783,53.636082,53.722608,52.818967,52.620637,52.744969,52.953783,53.636082,53.722608,2.1806e24,2.1600e24,2.1806e24,2.1600e24,1989.784411,6.935769
"""min""","""48951697""","""2026-02-01 02:01:59""","""2026-02-01 02:19:15""",13.0,0.0,0.0,0.0,0.00185,1.0,0.251776,"""209388000""","""ADAM SCHULTE""",0.0,0.0,"""-""",0.0,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-171.640658,-7.6811e24,-7.6811e24,-7.6811e24,-7.6811e24,3.855461,1.0
"""25%""",null,"""2026-02-06 19:54:16""","""2026-02-06 20:15:59""",92.0,8.638039,10.4,4.59,0.664313,1.012262,0.951759,null,null,9.7,70.0,null,null,7.407463,2.018814,4.305282,7.307409,9.832503,11.383014,7.407463,2.018814,4.305282,7.307409,9.832503,11.383014,7.407463,2.018814,4.305282,7.307409,9.832503,11.383014,89.390892,96.964259,89.390892,96.964259,2297.654349,6.0
"""50%""",null,"""2026-02-12 14:33:29""","""2026-02-12 14:48:00""",1298.0,12.253659,14.2,8.9,8.31821,1.02934,0.990594,null,null,11.9,70.0,null,null,12.561314,7.118336,9.760713,12.895038,16.456716,18.639275,12.561314,7.118336,9.760713,12.895038,16.456716,18.639275,12.561314,7.118336,9.760713,12.895038,16.456716,18.639275,98.091634,99.672838,98.091634,99.672838,3481.843185,12.0
"""75%""",null,"""2026-02-18 09:50:47""","""2026-02-18 09:57:18""",2299.0,14.705515,17.3,12.1,13.914301,1.093592,0.997318,null,null,13.1,70.0,null,null,17.545672,13.492098,15.946647,18.207955,20.228081,22.415436,17.545672,13.492098,15.946647,18.207955,20.228081,22.415436,17.545672,13.492098,15.946647,18.207955,20.228081,22.415436,99.626515,99.89737,99.626515,99.89737,5467.332835,18.0
"""max""","""49373155""","""2026-02-26 00:51:43""","""2026-02-26 01:15:50""",73134.0,33.360769,52.3,28.9,141.854579,38.307892,0.99918,"""636093273""","""YM THRONE""",18.0,89.0,"""tug""",1.0,31.875672,30.632685,31.319025,31.815379,32.501408,33.240559,31.875672,30.632685,31.319025,31.815379,32.501408,33.240559,31.875672,30.632685,31.319025,31.815379,32.501408,33.240559,99.994273,99.996302,99.994273,99.996302,17423.061877,26.0


In [41]:
bb_path = "s3://acoustic-sandbox/ambient-sound-analysis/data_2.0/broadband/hydrophone=orcasound_lab/year=2026/month=02/"
#bb_schema = pl.Schema({"__index_level_0__": dt.datetime,"0": pl.Float64})
bb_df = pl.scan_parquet(bb_path, storage_options={'aws_region': 'us-west-2'}).sort("__index_level_0__")

## Analysis

### Frequency of Ship Passings and Broadband

In [87]:
bb = (
    bb_df
    .group_by_dynamic("__index_level_0__", every="3h")
    .agg(pl.col("0").mean().alias("bb_avg"))
    .filter(pl.col("bb_avg") > -20)
    .sort("__index_level_0__")).collect()

In [73]:
ship_subset = ship_df.filter(pl.col("type").is_in(["cargo", "tanker"]))

In [93]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.3, 0.7],
    vertical_spacing=0.05
)
fig.add_trace(
go.Scatter(
    x=bb["__index_level_0__"],
    y=bb["bb_avg"],
    mode="lines",
    name="Broadband Sound Level"
), row=1, col=1)

# Duration in milliseconds
duration_ms = (ship_subset["l_timestamp"] - ship_subset["s_timestamp"]).dt.total_seconds() * 1000

fig.add_trace(
    go.Bar(
        x=duration_ms,
        y=ship_subset["type"],
        base=ship_subset["s_timestamp"],
        orientation="h",
        name="Ship Passages"
    ), row=2, col=1)

fig.update_layout(
    barmode="overlay",
    height=600,
    template="simple_white"
)

fig.update_xaxes(type="date")
fig.update_xaxes(showline=False, showticklabels=False, tickcolor="white", row=1, col=1)
fig.show()

### Ship Passage Timeline

In [ ]:
fig = px.timeline(ship_df, 
                    x_start="s_timestamp", 
                    x_end="l_timestamp", 
                    y="type", color="type", 
                    hover_data="bb_q95", 
                    template="simple_white", 
                    title="Ship Presence Timeline")
fig.show()

### Tanker Ship Crossing PSD

In [121]:
bb = (
    pa.bb_df
    .group_by_dynamic("__index_level_0__", every="1m")
    .agg(pl.col("0").mean().alias("bb_avg"))
    .sort("__index_level_0__")).collect()

In [128]:
start_time = dt.datetime(2026, 2, 11, 16, 43)
end_time = dt.datetime(2026, 2, 11, 17, 51)
psd_ship = pa2.psd_df.filter(pl.col("__index_level_0__").is_between(start_time, end_time)).collect()
bb_ship = pa2.bb_df.filter(pl.col("__index_level_0__").is_between(start_time, end_time)).collect()
ship_z = psd_ship.select(pl.exclude("__index_level_0__")).to_numpy().transpose()

In [129]:
fig = make_subplots(rows=2, cols=1, row_heights=[0.3,0.7], shared_xaxes=True, vertical_spacing=0.02)

fig.add_trace(go.Scatter(x=bb_ship["__index_level_0__"], y=bb_ship["0"], line=dict(color="#32006e", width=2)), row=1, col=1)

fig.add_trace(go.Heatmap(x=psd_ship["__index_level_0__"], y=psd_ship.columns, z=ship_z, zmin=0, zmax=30, colorscale='Viridis',
                    colorbar={"title": 'Magnitude'}), row=2, col=1)
    
fig.update_layout(
    legend_title="Magnitude",
    margin=dict(l=50, r=50, t=50, b=50),
    template="simple_white"
    )
fig.update_yaxes(title_text="Frequency (Hz)", type="log", row=2, col=1)
fig.update_yaxes(title_text="Decibels re 1 a.u.", row=1, col=1)
fig.update_xaxes(showline=False, showgrid=False, row=1, col=1)

### Cargo Ship Crossing

In [34]:
start_time = dt.datetime(2026, 2, 24, 19, 45)
end_time = dt.datetime(2026, 2, 24, 20, 50)
psd_ship = pa4.psd_df.filter(pl.col("__index_level_0__").is_between(start_time, end_time)).collect()
ship_z = psd_ship.select(pl.exclude("__index_level_0__")).to_numpy().transpose()

In [37]:
fig = go.Figure(
    data=go.Heatmap(x=psd_ship["__index_level_0__"], y=psd_ship.columns, z=ship_z, zmin=0, zmax=30, colorscale='Viridis',
                    colorbar={"title": 'Magnitude'}))
fig.update_layout(
    xaxis_title="Time",
    yaxis_title="Frequency (Hz)",
    legend_title="Magnitude",
    margin=dict(l=50, r=50, t=50, b=50)
    )
fig.update_yaxes(type="log")

### Cargo ships Speed vs Sound Level

In [ ]:
# cargo_df = (
#             ship_df
#             .filter(pl.col("type") == "cargo")
#             .filter(pl.col("bb_avg") > 0))
cargo_df = ship_df.filter(pl.col("bb_avg") > 0)
fig = go.Figure(
    data=go.Scatter(x=cargo_df["max_speed"], y=cargo_df["bb_q95"]*cargo_df["distance"], mode="markers", hovertext=cargo_df["name"])
)
fig.update_layout(
    xaxis_title="Speed (knots)",
    yaxis_title="Sound Level (Relative Decibels)",
    )


### Duration and Sound Level of Ships

In [137]:
fig = px.scatter(ship_df.filter(pl.col("bb_q95") > 0), x="duration", y="bb_q95", color="type")
fig.update_layout(
    xaxis_title="duration (s)",
    yaxis_title="Sound Level (Relative Decibels)",
    )
fig.show()